In [ ]:
import logging

import numpy as np
import openeo.processes
import shapely

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
import sys

sys.path.append("../utils/")

import utils

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
spatial_extent = {
    "west": 30.5503711040000994,
    "south": 1.0709279050000799,
    "east": 31.2229521229999989,
    "north": 1.5469373050000299,
}
temporal_extent = ["2019-10-01", "2025-04-01"]  # pad by ~3 months
bands = ["VH"]
instrument_mode = "IW"
orbit_state = "ascending"
relative_orbit = 101

speckle_filter_radius = 2
speckle_filter_cv_noise = 1 / np.sqrt(4)
speckle_filter_window_size = 5

resample_spatial_resolution = 30  # m

logistic_window_size = 11
logistic_steepness_parameter = -2.0  # dimensionless

temporal_variability_threshold = 0.5  # units: dB
flattening_threshold = 0.12  # units: dimensionless

logistic_sse_percentile = 0.95  # p95 is correct

min_connected_area = 10000  # m^2

In [ ]:
# very small test AOI
x = 30.944
y = 1.273
delta = 0.1
spatial_extent = {
    "west": x,
    "south": y,
    "east": x + delta,
    "north": y + delta,
}

In [ ]:
# spatial extent as dict of Polygon geometry
spatial_extent = shapely.geometry.mapping(
    shapely.box(
        xmin=spatial_extent["west"],
        ymin=spatial_extent["south"],
        xmax=spatial_extent["east"],
        ymax=spatial_extent["north"],
    )
)

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

## Sentinel 1

In [ ]:
# load results from previous batch job
JOB_ID = "j-260713124951418ebf4a620f7e421671"

s1_dB = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0000_s1_dB",
        },
    )
)

In [ ]:
# load_stac adds a time dimension 😠
s1_dB = s1_dB.drop_dimension("t")

In [ ]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0001_s1_dB",
        },
    )
)

In [ ]:
# unpack the bands into separate cubes
standard_deviation = s1_dB.filter_bands("sd").rename_labels(
    dimension="bands", target=bands
)
p05 = s1_dB.filter_bands("p05").rename_labels(dimension="bands", target=bands)
p95 = s1_dB.filter_bands("p95").rename_labels(dimension="bands", target=bands)
min_sse = s1_dB.filter_bands("min_sse").rename_labels(dimension="bands", target=bands)
min_sse_t = s1_dB.filter_bands("min_sse_t").rename_labels(
    dimension="bands", target=bands
)

In [ ]:
process_graph_results.append(
    standard_deviation.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_standard_deviation",
        },
    )
)
process_graph_results.append(
    p05.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_p05",
        },
    )
)
process_graph_results.append(
    p95.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_p95",
        },
    )
)
process_graph_results.append(
    min_sse.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_min_sse",
        },
    )
)
process_graph_results.append(
    min_sse_t.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_min_sse_t",
        },
    )
)

## temporal variability mask

In [ ]:
# mask of interesting pixels
temporal_variability_mask = standard_deviation >= temporal_variability_threshold

In [ ]:
process_graph_results.append(
    temporal_variability_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0020_temporal_variability_mask",
        },
    )
)
process_graph_results.append(
    temporal_variability_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_temporal_variability_mask",
        },
    )
)

# flattening mask

In [ ]:
# TODO: ATBD says denominator is abs(gamma_max)
# but code uses p05
# Dascalu 2023 has abs(gamma_max)
flattening = (p95 - p05) / p95.apply(openeo.processes.absolute)

In [ ]:
process_graph_results.append(
    flattening.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0030_flattening",
        },
    )
)
process_graph_results.append(
    flattening.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_flattening",
        },
    )
)

In [ ]:
# mask of good pixels where flattening >= flattening_threshold
flattening_mask = flattening >= flattening_threshold

In [ ]:
process_graph_results.append(
    flattening_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0040_flattening_mask",
        },
    )
)
process_graph_results.append(
    flattening_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_flattening_mask",
        },
    )
)

# load forest baseline

In [ ]:
# load results from previous batch job
JOB_ID = "j-2607100927534dd0aab0e65cccd05b80"

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0050_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_forest_baseline_mask",
        },
    )
)

In [ ]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

In [ ]:
# load_stac incorrectly sets nodata=0 😠
forest_baseline_mask = forest_baseline_mask.convert_data_type("bool")

In [ ]:
# make sure band dimension has consistent labels with other data cubes
forest_baseline_mask = forest_baseline_mask.rename_labels(
    dimension="bands", target=bands
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0052_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0052_forest_baseline_mask",
        },
    )
)

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
forest_baseline_mask = forest_baseline_mask.resample_cube_spatial(
    s1_dB,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0055_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0055_forest_baseline_mask",
        },
    )
)

# candidate mask

In [ ]:
# candidate pixel = 1
candidate_mask = temporal_variability_mask & flattening_mask & forest_baseline_mask

In [ ]:
process_graph_results.append(
    candidate_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0060_candidate_mask",
        },
    )
)
process_graph_results.append(
    candidate_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_candidate_mask",
        },
    )
)

In [ ]:
inverse_candidate_mask = utils.invert_mask(candidate_mask)

In [ ]:
process_graph_results.append(
    inverse_candidate_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0065_inverse_candidate_mask",
        },
    )
)
process_graph_results.append(
    inverse_candidate_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0065_inverse_candidate_mask",
        },
    )
)

## Mask based on percentile of min_sse

Here we discard candidate deforestation events where the SSE (goodness of fit) is above a certain percentile.

This is the final mask of deforestation event detections

In [ ]:
# only consider candidate pixels
min_sse_canditates = min_sse.mask(inverse_candidate_mask)

In [ ]:
process_graph_results.append(
    min_sse_canditates.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0070_min_sse_canditates",
        },
    )
)
process_graph_results.append(
    min_sse_canditates.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_min_sse_canditates",
        },
    )
)

In [ ]:
min_sse_at_percentile = utils.percentile_cube(
    cube=min_sse_canditates,
    aoi=spatial_extent,
    percentile=logistic_sse_percentile,
)
# mask of good pixels where min_sse <= logistic_sse_percentile
deforestation_event_mask = 0 <= (min_sse_at_percentile - min_sse_canditates)

In [ ]:
process_graph_results.append(
    deforestation_event_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0080_deforestation_event_mask",
        },
    )
)
process_graph_results.append(
    deforestation_event_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0080_deforestation_event_mask",
        },
    )
)

In [ ]:
inverse_deforestation_event_mask = utils.invert_mask(deforestation_event_mask)

In [ ]:
process_graph_results.append(
    inverse_deforestation_event_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0081_inverse_deforestation_event_mask",
        },
    )
)
process_graph_results.append(
    inverse_deforestation_event_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0081_inverse_deforestation_event_mask",
        },
    )
)

# Apply deforestation event mask to `min_sse_t`

In [ ]:
min_sse_t_masked = min_sse_t.mask(inverse_deforestation_event_mask)

In [ ]:
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0090_min_sse_t_masked",
        },
    )
)
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0090_min_sse_t_masked",
        },
    )
)

## mask based on connectivity

of the natural forest remaining, are regions of forest too small to meet the minimum connected area threshold?

In [ ]:
remaining_forest_mask = forest_baseline_mask & inverse_deforestation_event_mask

In [ ]:
process_graph_results.append(
    remaining_forest_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0100_remaining_forest_mask",
        },
    )
)
process_graph_results.append(
    remaining_forest_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0100_remaining_forest_mask",
        },
    )
)

In [ ]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        "pixel_area": resample_spatial_resolution * resample_spatial_resolution,
        "min_connected_area": min_connected_area,
    },
)

In [ ]:
# mask where 1 = small region to be excluded
small_region_mask = remaining_forest_mask.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0110_small_region_mask",
        },
    )
)

In [ ]:
# apply_neighborhood UDF seems to return float32, even if it's a mask
# data types: https://github.com/locationtech/geotrellis/blob/master/raster/src/main/scala/geotrellis/raster/CellType.scala
small_region_mask = small_region_mask.convert_data_type("bool")

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0111_small_region_mask",
        },
    )
)

In [ ]:
# stack min_sse_t_masked + small_region_mask into a single datacube
# in preparation for the nearest neighbour fill UDF

small_region_mask_renamed = small_region_mask.rename_labels(
    dimension="bands",
    target=["mask"],
)
min_sse_t_and_mask = min_sse_t_masked.merge_cubes(small_region_mask_renamed)

In [ ]:
process_graph_results.append(
    min_sse_t_and_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0120_min_sse_t_and_mask",
        },
    )
)
process_graph_results.append(
    min_sse_t_and_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0120_min_sse_t_and_mask",
        },
    )
)

In [ ]:
nearest_neighbour_fill_udf = openeo.UDF.from_file(
    "../udf/nearest_neighbour_fill.py",
    runtime="Python",
    version="3.11",
)

In [ ]:
min_sse_t_and_mask = min_sse_t_and_mask.apply_neighborhood(
    nearest_neighbour_fill_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    min_sse_t_and_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0130_min_sse_t_and_mask",
        },
    )
)
process_graph_results.append(
    min_sse_t_and_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0130_min_sse_t_and_mask",
        },
    )
)

In [ ]:
# min_sse_t_and_mask (as returned from the nearest_neighbour_fill_udf) still has 2 bands
# drop the second band, which is the small_region_mask
min_sse_t_masked = min_sse_t_and_mask.filter_bands(bands)

In [ ]:
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0140_min_sse_t_masked",
        },
    )
)
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0140_min_sse_t_masked",
        },
    )
)

# Run batch job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)

In [ ]:
job = multi_result.create_job()
job.start_and_wait()
# Inspect job.logs() if it fails

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)